# Task 1 - Smart Buildings: Dataset Understanding and Probability

**Member 1 - Data Lead**

> Member 5: replace this starter with the final problem statement, objectives, and roadmap before submission.

This notebook analyzes the Smart Buildings **simulation-run log** (`combined_runsfinal.csv`): 60 simulated two-week runs (6 buildings x 5 weather files x 2 control policies), recorded every 5 minutes. It documents the source file, defines a reproducible analysis frame, checks data quality, and publishes shared event columns for the probability applications. Later sections are reserved for the team. Results describe these simulated runs, not real-world buildings.

## 1. Setup (Member 1)

Load the single source CSV without editing it. The notebook first checks its current folder and the usual shared Drive folder. If the file is not there and this is Colab, it asks you to upload it. If running locally, keep this notebook beside the CSV or set `DATA_DIR` to its folder.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

REQUIRED_FILE = 'combined_runsfinal.csv'
candidate_dirs = [Path.cwd(), Path('/content'), Path('/content/drive/MyDrive/stats_project')]
DATA_DIR = next((p for p in candidate_dirs if (p / REQUIRED_FILE).exists()), None)

if DATA_DIR is None:
    try:
        from google.colab import files
        print(f'Upload {REQUIRED_FILE}')
        files.upload()
        DATA_DIR = Path.cwd()
    except ImportError:
        raise FileNotFoundError(
            f'Could not find {REQUIRED_FILE}. Put it beside this notebook or change DATA_DIR to its folder.'
        )

if not (DATA_DIR / REQUIRED_FILE).exists():
    raise FileNotFoundError(f'Missing {REQUIRED_FILE} in {DATA_DIR}.')

df_raw = pd.read_csv(DATA_DIR / REQUIRED_FILE)       # never modified after this line

print('Data folder:', DATA_DIR)
print('step-level table:', df_raw.shape)
print('runs:', df_raw.run_id.nunique(), '| buildings:', df_raw.building.nunique(),
      '| weather files:', df_raw.weather.nunique(), '| policies:', df_raw.policy.nunique())
assert df_raw.shape == (241920, 37)
assert df_raw.run_id.nunique() == 60

## 2. Understand the file and variables (Member 1)

Each row is **one 5-minute time step of one simulated run**. A run is one building, driven by one weather file, under one control policy; its `run_id` is `building__weather__policy`. The 60 runs are 6 buildings x 5 weather files per building x 2 policies (`pid_water` and `noisy_pid_water`). Every run has exactly 4,032 steps (14 days), so the file has 60 x 4,032 = 241,920 rows.

| Level | Columns | What they hold |
|---|---|---|
| Identifiers | `run_id`, `building`, `weather`, `policy` | Which run a row belongs to (`run_id` + `step` is the row key) |
| Time | `step`, `timestamp` | Step counter 0-4031 and the simulated clock time |
| Step observations | `occupancy`, `outside_air_temp_K`, `zone_temp_mean_K`, `zone_temp_min_K`, `zone_temp_max_K` | What the building and weather looked like at that step |
| Step reward | `reward`, `done`, `rc_*` | The reward signal and its cost/productivity/carbon components |
| Run summaries | `run_*` | One value per run, repeated on every row of that run (building size, run totals, run time) |

Building structure (`run_n_zones`, `run_n_floors`, `run_n_ahus`, `run_n_obs`, ...) is identical for all runs of the same building, so the six buildings are the only distinct structures in this file. The probability sections therefore treat *building* and *run* as the units that carry structure, and *steps* as the units that carry weather, occupancy, and comfort.

In [ ]:
# A small readable sample: one run's first steps, then one row per building.
sample_cols = ['run_id', 'step', 'timestamp', 'occupancy', 'outside_air_temp_K',
               'zone_temp_mean_K', 'reward', 'run_n_zones', 'run_n_floors']
display(df_raw.loc[df_raw.run_id == df_raw.run_id.iloc[0], sample_cols].head(5))
display(df_raw.drop_duplicates('building')[['building', 'run_n_zones', 'run_n_floors', 'run_n_ahus',
                                              'run_n_obs', 'run_n_cont_actions', 'run_n_disc_actions']]
        .reset_index(drop=True))

### Variable dictionary

The next table includes every CSV field. `Meaning` is a plain-English guide inferred from the column name and its values (this file ships without a feature catalog). `level` is measured from the data: *constant* if it never changes in the whole file, *building-level* if it never changes within a building, *run-level* if it never changes within a run, otherwise *step-level*.

In [ ]:
meanings = {
    'run_id': 'Run label: building__weather__policy', 'building': 'Building used in the run',
    'weather': 'Weather file used (city_number)', 'policy': 'Control policy (pid_water or noisy_pid_water)',
    'step': 'Step counter within the run (0-4031, one step = 5 minutes)', 'timestamp': 'Simulated clock time of the step',
    'reward': 'Reward the agent received at this step (<= 0 here)', 'occupancy': 'People present in the building at this step',
    'outside_air_temp_K': 'Outdoor air temperature (kelvin)', 'zone_temp_mean_K': 'Mean zone temperature across all zones (kelvin)',
    'zone_temp_min_K': 'Coldest zone temperature (kelvin)', 'zone_temp_max_K': 'Hottest zone temperature (kelvin)',
    'done': 'Episode-finished flag', 'rc_agent_reward': 'Agent reward as logged by the reward component tracker',
    'rc_productivity_reward': 'Productivity part of the reward', 'rc_electricity_energy_cost': 'Electricity cost at this step',
    'rc_natural_gas_energy_cost': 'Natural-gas cost at this step', 'rc_carbon_emitted': 'Carbon emitted at this step',
    'rc_normalized_productivity_regret': 'Productivity regret, normalized', 'rc_normalized_energy_cost': 'Energy cost, normalized',
    'rc_normalized_carbon_emission': 'Carbon emission, normalized', 'rc_total_occupancy': 'Occupancy as logged by the reward tracker',
    'run_status': 'Whether the run finished correctly', 'run_steps': 'Number of steps in the run',
    'run_total_reward': 'Sum of reward over the run', 'run_mean_reward': 'Mean reward per step over the run',
    'run_score_abs_mean': 'Absolute value of the mean reward', 'run_avg_occupancy': 'Mean occupancy over the run',
    'run_avg_zone_temp_K': 'Mean of the mean-zone temperature over the run', 'run_n_obs': 'Length of the observation vector',
    'run_n_cont_actions': 'Number of continuous action dimensions', 'run_n_disc_actions': 'Number of discrete action dimensions',
    'run_n_ahus': 'Number of air handlers', 'run_n_zones': 'Number of thermal zones', 'run_n_floors': 'Number of floors',
    'run_size_mb': 'Size of the stored run (MB)', 'run_seconds': 'Wall-clock seconds the run took',
}

def classify_type(column, series):
    if column == 'timestamp':
        return 'date/time'
    if column in ('run_id', 'building', 'weather', 'policy', 'run_status'):
        return 'categorical/identifier'
    if pd.api.types.is_integer_dtype(series):
        return 'numeric-discrete'
    return 'numeric-continuous'

def classify_level(column):
    by_run = df_raw.groupby('run_id')[column].nunique(dropna=False).max() == 1
    by_building = df_raw.groupby('building')[column].nunique(dropna=False).max() == 1
    if column in ('run_id',):
        return 'run key'
    if df_raw[column].nunique(dropna=False) == 1:
        return 'constant (whole file)'
    if by_building:
        return 'building-level'
    if by_run:
        return 'run-level'
    return 'step-level'

assert set(meanings) == set(df_raw.columns), set(df_raw.columns) ^ set(meanings)
variable_dictionary = pd.DataFrame([
    {'column': c, 'level': classify_level(c), 'type': classify_type(c, df_raw[c]), 'meaning': meanings[c]}
    for c in df_raw.columns
])
assert len(variable_dictionary) == 37
display(variable_dictionary)
print(variable_dictionary.level.value_counts().to_string())

## 3. Minimal preprocessing and quality checks (Member 1)

The source data is already clean: we expect no missing values, no duplicate rows, and a complete 4,032-step run for every `run_id`, so no observations need to be fixed or removed. Keep `df_raw` unchanged. In the analysis copy `df` we (a) parse `timestamp`, (b) add `city` (the first part of `weather`) and `hour` (hour of day), and (c) omit columns that are constant over the whole file because they cannot define a non-trivial event. Valid extremes, such as very cold Fairbanks nights or the 85-zone building, are kept.

The checks below also confirm the file is internally consistent: run summaries match what we recompute from the step rows, rewards and temperatures agree between columns, and the two policies in each building/weather pair see the same weather and occupancy. All probability results on `df` describe these simulated runs.

In [ ]:
preprocess_log = []
def log_step(name, frame):
    preprocess_log.append({'step': name, 'rows': len(frame), 'columns': frame.shape[1]})
    print(f"{name}: {len(frame):,} rows x {frame.shape[1]} columns")

log_step('Raw step-level file', df_raw)
df = df_raw.copy()
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['city'] = df['weather'].str.split('_').str[0]
df['hour'] = df['timestamp'].dt.hour
log_step('Parsed timestamp, added city and hour', df)
const_cols = [c for c in df.columns if df[c].nunique(dropna=False) == 1]
df = df.drop(columns=const_cols)
log_step(f'Dropped {len(const_cols)} constant columns (analysis frame only)', df)

# --- Quality checks (shared data contract) -------------------------------------------------
g = df.groupby('run_id')
run_cols = [c for c in df_raw.columns if c.startswith('run_') and c != 'run_id' and c in df.columns]
building_cols = ['run_n_zones', 'run_n_floors', 'run_n_ahus', 'run_n_obs', 'run_n_cont_actions', 'run_n_disc_actions']
checks = {}
checks['missing cells in raw file'] = int(df_raw.isna().sum().sum())
checks['duplicate raw rows'] = int(df_raw.duplicated().sum())
checks['duplicate (run_id, step) keys'] = int(df_raw.duplicated(['run_id', 'step']).sum())
checks['runs'] = int(df.run_id.nunique())
checks['every run has 4,032 steps numbered 0-4031'] = bool(
    (g.step.count() == 4032).all() and (g.step.min() == 0).all() and (g.step.max() == 4031).all())
checks['run_id equals building__weather__policy'] = bool(
    (df.run_id == df.building + '__' + df.weather + '__' + df.policy).all())
checks['every building/weather pair has both policies'] = bool(
    (df.groupby(['building', 'weather']).policy.nunique() == 2).all())
checks['timestamps advance by exactly 5 minutes inside every run'] = bool(
    (g.timestamp.diff().dropna() == pd.Timedelta(minutes=5)).all())
checks['run_* columns constant within each run'] = bool((g[run_cols].nunique() == 1).all().all())
checks['building structure constant within each building'] = bool(
    (df.groupby('building')[building_cols].nunique() == 1).all().all())
checks['reward equals rc_agent_reward'] = bool(np.allclose(df.reward, df.rc_agent_reward, atol=1e-12))
checks['zone_temp_min <= mean <= max'] = bool(
    ((df.zone_temp_min_K <= df.zone_temp_mean_K + 1e-9) & (df.zone_temp_mean_K <= df.zone_temp_max_K + 1e-9)).all())
rec = g.agg(total=('reward', 'sum'), mean=('reward', 'mean'), occ=('occupancy', 'mean'), temp=('zone_temp_mean_K', 'mean'),
            t0=('run_total_reward', 'first'), m0=('run_mean_reward', 'first'),
            o0=('run_avg_occupancy', 'first'), z0=('run_avg_zone_temp_K', 'first'))
checks['run summaries match recomputed values (tol 1e-3)'] = bool(
    np.allclose(rec.total, rec.t0, atol=1e-3) and np.allclose(rec['mean'], rec.m0, atol=1e-3)
    and np.allclose(rec.occ, rec.o0, atol=1e-3) and np.allclose(rec.temp, rec.z0, atol=1e-3))
checks['run_score_abs_mean = |run_mean_reward|'] = bool(np.allclose(df.run_score_abs_mean, df.run_mean_reward.abs()))
# the two policies in a building/weather pair are driven by the same weather and the same occupants
key = ['building', 'weather', 'step']
pair = df.pivot_table(index=key, columns='policy', values=['outside_air_temp_K', 'occupancy'])
checks['paired policies see identical weather and occupancy'] = bool(
    (pair['outside_air_temp_K']['pid_water'] == pair['outside_air_temp_K']['noisy_pid_water']).all()
    and (pair['occupancy']['pid_water'] == pair['occupancy']['noisy_pid_water']).all())

assert checks['missing cells in raw file'] == 0 and checks['duplicate raw rows'] == 0
assert checks['duplicate (run_id, step) keys'] == 0 and checks['runs'] == 60
assert all(v for k, v in checks.items() if isinstance(v, bool)), [k for k, v in checks.items() if v is False]
assert set(const_cols) == {'done', 'run_status', 'run_steps'}, const_cols

display(pd.Series(checks, name='result').to_frame())
display(pd.DataFrame(preprocess_log))
print('Constant columns (kept in df_raw):', const_cols)

# --- One observation worth knowing about (documented, not corrected) ----------------------
diff = (df.occupancy != df.rc_total_occupancy)
print(f"\noccupancy differs from rc_total_occupancy on {diff.mean():.1%} of steps; "
      f"run totals: occupancy = {int(df.loc[df.run_id == df.run_id.iloc[0], 'occupancy'].sum()):,}, "
      f"rc_total_occupancy = {int(df.loc[df.run_id == df.run_id.iloc[0], 'rc_total_occupancy'].sum()):,} (first run)")
print('-> the analysis uses `occupancy`; rc_total_occupancy is kept but treated as a separate reward-tracker log.')

## 4. Run table and shared event columns (Member 1)

Build `runs`, one row per run (60 rows), from the first row of each `run_id`. Then publish the following reusable Boolean columns without renaming. **Building / run events** (also stored in `runs`): `A_large`, `B_noisy`, `F2`, `F3`, `F4`. **Step events** (in `df` only): `OCC`, `COLD`, `UNCOMF`.

| Event | Definition | Level |
|---|---|---|
| `A_large` | `run_n_zones` > median zone count of the 60 runs (46.5, so no ties); the 3 largest buildings | building / run / step |
| `B_noisy` | policy is `noisy_pid_water` | run / step |
| `F2`, `F3`, `F4` | building has 2, 3 or 4 floors | building / run / step |
| `OCC` | `occupancy` > 0 (someone is in the building) | step |
| `COLD` | `outside_air_temp_K` < 283.15 K (below 10 °C) | step |
| `UNCOMF` | `zone_temp_mean_K` outside 293.15-299.15 K (20-26 °C) | step |

The comfort band and the 10 °C cutoff are conventions chosen for this project, not values stored in the data. The `A_large` threshold uses a strict greater-than comparison against the median, as in the original notebook.

In [ ]:
id_cols = ['run_id', 'building', 'weather', 'city', 'policy']
runs = (df.drop_duplicates('run_id')[id_cols + run_cols].reset_index(drop=True))
assert len(runs) == 60

zone_median = runs.run_n_zones.median()                     # 46.5
assert zone_median not in set(runs.run_n_zones)             # no ties at the threshold

def add_run_events(frame):
    frame['A_large'] = frame['run_n_zones'] > zone_median
    frame['B_noisy'] = frame['policy'] == 'noisy_pid_water'
    for k in (2, 3, 4):
        frame[f'F{k}'] = frame['run_n_floors'] == k
    return frame

runs = add_run_events(runs)
df = add_run_events(df)
df['OCC']    = df['occupancy'] > 0
df['COLD']   = df['outside_air_temp_K'] < 283.15
df['UNCOMF'] = (df['zone_temp_mean_K'] < 293.15) | (df['zone_temp_mean_K'] > 299.15)

RUN_EVENTS  = ['A_large', 'B_noisy', 'F2', 'F3', 'F4']
STEP_EVENTS = RUN_EVENTS + ['OCC', 'COLD', 'UNCOMF']
for col in STEP_EVENTS:
    assert col in df.columns and df[col].dtype == bool, col

# consistency: the run-level events agree with the step-level ones
step_side = df.groupby('run_id')[RUN_EVENTS].all().loc[runs.run_id]      # same row order as runs
assert (step_side.to_numpy() == runs[RUN_EVENTS].to_numpy()).all()
assert (df.F2.astype(int) + df.F3.astype(int) + df.F4.astype(int) == 1).all()   # floors partition the rows

event_table = pd.DataFrame({
    'run-level count (of 60)': runs[RUN_EVENTS].sum().reindex(STEP_EVENTS),
    'step-level count': df[STEP_EVENTS].sum(),
    'step-level probability': df[STEP_EVENTS].mean(),
})
display(event_table.round(4))
print('Rows in df:', f'{len(df):,}', '| shared event columns:', STEP_EVENTS)
print('Buildings per floor count:', runs.drop_duplicates('building').run_n_floors.value_counts().sort_index().to_dict())
print('Large buildings:', sorted(runs.loc[runs.A_large, 'building'].unique()))

# The step probabilities are exact counts over this file; fix the ones other members rely on
assert int(df.A_large.sum()) == 120_960 and int(df.B_noisy.sum()) == 120_960
assert int(df.F2.sum()) == 120_960 and int(df.F3.sum()) == 80_640 and int(df.F4.sum()) == 40_320

### Preprocessing notes and limitations (Member 1)

No rows were removed: the analysis retains all 241,920 steps of all 60 runs. Constant columns (`done`, `run_status`, `run_steps`) were removed only from `df`, not from `df_raw`. Some columns carry the same information: `reward` equals `rc_agent_reward`, `run_score_abs_mean` is `|run_mean_reward|`, and the building-structure columns (`run_n_zones`, `run_n_obs`, `run_n_cont_actions`, ...) move together. `run_size_mb` and `run_seconds` describe the run, not the building's behaviour, and should not define events.

These are simulator outputs, not real buildings, so the main cautions are about what the 241,920 rows really contain:

- **Only 6 distinct buildings.** Anything that depends on size or floors (`A_large`, `F2`-`F4`) is decided by 6 building types, so the 241,920 rows are not 241,920 independent structures. `F4` is a single building.
- **Steps are not independent draws.** Consecutive 5-minute steps are strongly autocorrelated (weather, occupancy and temperatures change slowly), so step-level probabilities are accurate frequencies in this file, but their sampling uncertainty is governed by the 60 runs (and 30 distinct building/weather pairs), not by 241,920.
- **The two policies are paired, not independent.** Each building/weather pair is run once under each policy with identical weather and occupancy, so comparisons of `B_noisy` with weather or occupancy events are exact by design.
- **Weather is not fully crossed with buildings.** Each building sees 5 specific weather files (one per city) for a single 14-day window, so a city's weather is confounded with the building it was paired with.
- **Occupancy follows a daily schedule.** `occupancy` is 0 from 19:00 to 05:59 and peaks around midday, so `OCC` is largely a time-of-day event. `occupancy` and `rc_total_occupancy` disagree on about a third of steps; we use `occupancy`.
- **Conventions, not facts.** The median split for `A_large`, the 10 °C cutoff for `COLD` and the 20-26 °C band for `UNCOMF` are project choices, not physical definitions.
- **Only two controllers.** Both policies are PID water-loop variants (one with added noise), so nothing here says how other control strategies would behave.

> **Update for the new data (Member 1) - read before running Sections 5-7.** The analysis frame is now the simulation-run log, so the objects the later sections were written for no longer exist: `df_raw` is now the 241,920-row step table, and `rej_raw`, `cat_raw`, `seeds` and the 200-building frame are gone. The shared event columns changed as follows.
>
> | Kept (same name, new meaning) | New | Removed |
> |---|---|---|
> | `A_large`, `F2`, `F3`, `F4` | `B_noisy`, `OCC`, `COLD`, `UNCOMF` | `B_extra_ahu`, `O2`, `N7`, and the rejected-seed model |
>
> The hard-coded counts, thresholds and quoted numbers in Sections 5-6 (for example "98 large buildings", "146 extra-AHU buildings", `assert ... == 200`) belong to the old data and must be replaced by Members 2-4 using the new events and the two frames `df` (step level) and `runs` (run level).

## 5. Probability applications (Members 2-4)

Team members: add each application using the required five-part template: question; events and assumptions; rule and why it fits; calculation and result; interpretation. Keep this section after the shared event-column cell.

### 5.0 Member 2 setup: Probability A (foundations and counting)

Applications 1-5 below cover sample spaces and events, the axioms, the complement rule, counting techniques, the addition rule, and a second probability model on the rejected seeds. Each application follows the same five parts: **question; events and assumptions; rule and why it fits; calculation and result; interpretation**.

This cell checks that Member 1's shared columns exist before we build on them, fixes the random seed (42), and creates `m2_results`, the dictionary of rule-based values handed to Member 4 for validation.

In [ ]:
from math import comb, perm
from itertools import combinations
import json, os
import matplotlib.pyplot as plt

RANDOM_SEED = 42                      # fixed seed: reruns give identical numbers
rng = np.random.default_rng(RANDOM_SEED)
PALETTE = {'main': '#2F5D8C', 'accent': '#D9822B', 'grey': '#8A8F98'}
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titlesize': 12, 'axes.labelsize': 10})
os.makedirs('figures', exist_ok=True)

# Confirm Member 1's data contract is intact (names must not change).
events = ['A_large', 'B_extra_ahu', 'F2', 'F3', 'F4', 'O2']
for col in events:
    assert col in df.columns and df[col].dtype == bool, f'Problem with event column {col}'
print(df[events].sum().to_string())
print('seeds table:', seeds.shape, '| accepted:', int(seeds.accepted.sum()), '| rejected:', int((~seeds.accepted).sum()))
print('rejected-seed table:', rej.shape)

m2_results = {}                       # rule-based results handed to Member 4

## 5.1 Sample space, events, axioms and the complement rule (Member 2)

### Application 1 - Sample space and events: how often does the generator reject a seed?

**Question.** What is the probability that a seed tried by the generator is rejected?

**Events and assumptions.** The sample space **S** is the 282 seeds 100000-100281, every seed the generator tried. Event **R** = "seed rejected" (the minimum-zone-area check failed). Assumption: the seeds are treated as equally likely draws from the generator. The generator is deterministic, so this is a modelling assumption and the probability is a relative frequency over this seed range.

**Rule and why it fits.** For equally likely outcomes in a finite sample space, P(R) = |R| / |S|. We use S = 282 and not 200: the 200 buildings are only the accepted outcomes, so using them would answer a different question.

In [ ]:
S_size = len(seeds)                      # |S|
R_mask = ~seeds.accepted                 # True for rejected seeds
R_size = int(R_mask.sum())               # |R|
p_R = R_size / S_size

# Independent check: count the rejected seeds two other ways.
assert R_size == len(rej_raw) == len(set(range(100000, 100282)) - set(df.seed))

print(f'|S| = {S_size}, |R| = {R_size}')
print(f'P(R) = {R_size}/{S_size} = {p_R:.4f}')

# Practical use: seeds needed for N usable buildings = N / P(accepted)
p_acc = 1 - p_R
for N in (200, 1000):
    print(f'To get {N} usable buildings, try about {N / p_acc:.0f} seeds')
m2_results['P(R)'] = p_R

**Result.** P(R) = 82/282 = 0.291.

**Interpretation.** About 29% of the seeds the generator tries are wasted; only about 71% give a usable building. To obtain N buildings, organizers should plan to try about N / 0.709 seeds (about 1,410 for 1,000 buildings). The rate belongs to this seed range and is not a universal constant of the generator. The run stopped at the 200th accepted building, so the last seed is accepted by construction; the effect on 82/282 is negligible.

### Application 2 - Axioms and the complement rule

**Question.** (a) Is the empirical distribution of `num_floors` a valid probability model? (b) What is the probability that a building needs *no* extra AHU?

**Events and assumptions.** The sample space is the 200 accepted buildings (conditional on acceptance). Outcomes are the floor counts 2, 3, 4 with F2, F3, F4 as events. B = "the scenario generator added AHUs beyond the configured number".

**Rules and why they fit.**
- *Axioms:* (1) P(E) >= 0, (2) P(S) = 1, (3) P(E1 or E2) = P(E1) + P(E2) when E1 and E2 are disjoint. A set of numbers is a valid probability model only if it satisfies all three, so we test each one.
- *Complement rule:* P(not B) = 1 - P(B). It fits because "no extra AHU" is exactly the complement of B, and it is easier to compute from P(B) than to build separately. We confirm it with two direct counts.

In [ ]:
floors = df['floor_plan.num_floors']
p_floor = floors.value_counts(normalize=True).sort_index()
print(p_floor.round(3).to_string())

# Axiom 1: every probability is non-negative
assert (p_floor >= 0).all()
# Axiom 2: the probabilities over the whole sample space add to 1
print('Sum over S =', p_floor.sum())
assert np.isclose(p_floor.sum(), 1.0)
# Axiom 3: additivity for disjoint events (F2 and F3 cannot both happen)
assert (df.F2 & df.F3).sum() == 0
print('P(F2 or F3) =', (df.F2 | df.F3).mean(), ' vs  P(F2)+P(F3) =', p_floor[2] + p_floor[3])
assert np.isclose((df.F2 | df.F3).mean(), p_floor[2] + p_floor[3])

# Complement rule, checked three ways
p_B = df.B_extra_ahu.mean()
p_notB_rule   = 1 - p_B                                  # rule
p_notB_count  = (~df.B_extra_ahu).mean()                 # direct count of the complement
p_notB_config = (df.n_ahus == df['ahu.num_ahus']).mean() # independent: AHU count equals configured count
print(f'P(B) = {p_B:.3f}')
print(f'P(not B): rule = {p_notB_rule:.3f} | count = {p_notB_count:.3f} ({int((~df.B_extra_ahu).sum())}/200) | n_ahus==configured = {p_notB_config:.3f}')
assert np.isclose(p_notB_rule, p_notB_count) and np.isclose(p_notB_count, p_notB_config)

m2_results.update({'P(B)': p_B, 'P(not B)': p_notB_rule,
                   'P(F2)': p_floor[2], 'P(F3)': p_floor[3], 'P(F4)': p_floor[4]})

fig, ax = plt.subplots(figsize=(5.2, 3.6))
bars = ax.bar([str(k) for k in p_floor.index], p_floor.values, color=PALETTE['main'], width=0.6)
for b, v in zip(bars, p_floor.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.008, f'{v:.3f}', ha='center', fontsize=10)
ax.set_ylim(0, 0.45); ax.set_xlabel('Number of floors'); ax.set_ylabel('Probability')
ax.set_title('P(num_floors) over the 200 accepted buildings')
plt.tight_layout(); plt.savefig('figures/fig_floors_pmf.png'); plt.show()

**Result.** P(2) = 0.370, P(3) = 0.315, P(4) = 0.315; all are non-negative, they sum to 1, and F2 and F3 are disjoint so their probabilities add. P(not B) = 1 - 0.73 = 0.27 (54 of 200), identical by the rule, by the direct count, and by checking that `n_ahus` equals the configured AHU count.

**Interpretation.** The empirical floor distribution is a valid probability model. Only about 1 building in 4 stays within its configured AHU count; the other 73% need the scenario generator to add air handlers.

## 5.2 Counting techniques (Member 2)

### Application 3 - Counting: combinations, permutations, the multiplication principle

**Question.** How many ways can we select or arrange buildings, and what does that tell us about drawing review samples, assigning train/validation/test roles and covering the configuration space?

**Events and assumptions.** Buildings are distinguishable (each has its own seed), the 200 accepted buildings are the population, and every selection is equally likely. Counting-based probability = favourable selections / all selections.

**Rules and why they fit.**
- *Combinations* C(n, k) = n! / (k!(n-k)!): order does not matter when a reviewer picks 3 buildings to inspect. P(all 3 needed extra AHUs) = C(146,3) / C(200,3), sampling **without replacement** because the same building cannot be picked twice.
- *Permutations* P(n, k) = n! / (n-k)!: order matters when the first building is for train, the second for validation, the third for test.
- *Multiplication principle:* independent choices multiply, so 3 floor options x 5 AHU options x 2 occupant options = 30 configuration cells.
- *Train/test splits:* the number of 140/60 splits is C(200, 140).

In [ ]:
n = len(df); n_B = int(df.B_extra_ahu.sum())

# (a) Combinations: 3 buildings picked at random, all needed extra AHUs
p_all3 = comb(n_B, 3) / comb(n, 3)
p_approx = (n_B / n) ** 3                       # with-replacement approximation
p_seq = (n_B/n) * ((n_B-1)/(n-1)) * ((n_B-2)/(n-2))   # picking one at a time, no replacement
print(f'C({n_B},3) / C({n},3) = {comb(n_B,3)} / {comb(n,3)} = {p_all3:.4f}')
print(f'with-replacement approximation 0.73^3 = {p_approx:.4f}  (gap {p_approx - p_all3:+.4f})')
print(f'one-at-a-time product              = {p_seq:.4f}')

# Independent check: enumerate every one of the 1,313,400 triples and count directly
flags = df.B_extra_ahu.tolist()
good = sum(1 for i, j, k in combinations(range(n), 3) if flags[i] and flags[j] and flags[k])
print('Enumerated triples:', comb(n, 3), '| all-three-extra-AHU triples:', good, '| P =', round(good / comb(n, 3), 4))
assert good == comb(n_B, 3) and np.isclose(p_all3, p_seq)

# (b) Permutations vs combinations
print(f'Ordered (train, val, test) triples  P(200,3) = {perm(n,3):,}')
print(f'Unordered triples                   C(200,3) = {comb(n,3):,}')
print(f'Ratio = {perm(n,3) // comb(n,3)} = 3! (the orderings of one triple)')

# (c) Multiplication principle on the configuration space
levels = {'floors': sorted(floors.unique()),
          'num_ahus': sorted(df['ahu.num_ahus'].unique()),
          'occupants': sorted(df['simulation.occupants_per_zone'].unique())}
n_cells = int(np.prod([len(v) for v in levels.values()]))
print('Option counts:', {k: len(v) for k, v in levels.items()}, '->', n_cells, 'possible cells')
cell_counts = df.groupby(['floor_plan.num_floors', 'ahu.num_ahus', 'simulation.occupants_per_zone']).size()
full_index = pd.MultiIndex.from_product(levels.values())
cell_counts = cell_counts.reindex(full_index, fill_value=0)
print(f'Cells observed: {(cell_counts > 0).sum()} of {n_cells}; empty cells: {(cell_counts == 0).sum()}')
print(f'Buildings per cell: min {cell_counts.min()}, max {cell_counts.max()}, uniform expectation {n/n_cells:.1f}')
display(cell_counts.unstack(level=-1).rename_axis(['floors', 'num_ahus']).add_prefix('occupants='))

# (d) Train/test splits
n_splits = comb(200, 140)
assert n_splits == comb(200, 60)
print(f'Possible 140/60 splits: C(200,140) has {len(str(n_splits))} digits (about {n_splits:.2e})')

m2_results.update({'P(all 3 extra AHU, no replacement)': p_all3, 'P(all 3, with-replacement approx)': p_approx,
                   'perm(200,3)': perm(n,3), 'comb(200,3)': comb(n,3), 'config cells observed': int((cell_counts>0).sum())})

**Result.** P(all 3 picked buildings needed extra AHUs) = C(146,3)/C(200,3) = 0.3868. The with-replacement shortcut 0.73^3 = 0.3890 is slightly larger. Enumerating all 1,313,400 triples gives exactly the same count as the formula. There are 7,880,400 ordered (train, validation, test) triples versus 1,313,400 unordered ones (a factor of 3! = 6). All 30 configuration cells occur (2 to 12 buildings each), and there are about 10^52 possible 140/60 splits.

**Interpretation.**
- *Why the small gap:* when a building is picked, it cannot be picked again, so the proportion of "extra-AHU" buildings left drops a little with each pick. With replacement ignores that. The gap is small because 3 is tiny compared with 200.
- *Order:* use combinations when only *which* buildings matter (a review sample) and permutations when *roles* matter (train/validation/test).
- *Coverage:* the sample covers every configuration cell, but with few buildings in some cells (as few as 2), so cell-level statements are noisy.
- *Splits:* the number of possible splits is astronomically large, which is why one fixed split with a fixed seed must be reported to make results reproducible.

## 5.3 The addition rule (Member 2)

### Application 4 - Addition rule: general case and mutually exclusive case

**Question.** (a) What is the probability that a building is large **or** needs an extra AHU? (b) What is the probability that a building has 2 floors **or** 4 floors?

**Events and assumptions.** A = n_zones > 64 (the median), B = extra AHU. A and B can both happen, so they overlap. F2 and F4 describe the same variable and cannot both be true.

**Rules and why they fit.**
- *General addition rule:* P(A or B) = P(A) + P(B) - P(A and B). It fits because A and B overlap; plain addition would count the buildings in both events twice.
- *Mutually exclusive case:* P(F2 or F4) = P(F2) + P(F4), because P(F2 and F4) = 0 and there is nothing to subtract.

In [ ]:
A, B = df.A_large, df.B_extra_ahu
p_A, p_B, p_AB = A.mean(), B.mean(), (A & B).mean()

p_or_rule   = p_A + p_B - p_AB                 # general addition rule
p_or_direct = (A | B).mean()                   # direct count: rows where A or B is True
print(f'P(A)={p_A:.3f}  P(B)={p_B:.3f}  P(A and B)={p_AB:.3f}')
print(f'Rule:   {p_A:.3f} + {p_B:.3f} - {p_AB:.3f} = {p_or_rule:.3f}')
print(f'Direct: {int((A | B).sum())}/200 = {p_or_direct:.3f}')
print(f'Naive sum without subtracting = {p_A + p_B:.3f}  (impossible: above 1)')
assert np.isclose(p_or_rule, p_or_direct)

# Complement cross-check: neither large nor extra AHU
print('P(neither) =', round(1 - p_or_rule, 3), '| direct count =', int((~A & ~B).sum()), 'buildings')

# Mutually exclusive case
assert (df.F2 & df.F4).sum() == 0
p_2or4_rule = df.F2.mean() + df.F4.mean()
p_2or4_direct = (df.F2 | df.F4).mean()
print(f'P(F2 or F4): rule {p_2or4_rule:.3f} | direct {p_2or4_direct:.3f} | overlap count {(df.F2 & df.F4).sum()}')
assert np.isclose(p_2or4_rule, p_2or4_direct)

m2_results.update({'P(A and B)': p_AB, 'P(A or B)': p_or_rule, 'P(F2 or F4)': p_2or4_rule})

**Result.** P(A or B) = 0.49 + 0.73 - 0.485 = 0.735 (147 of 200 buildings), matching the direct count. P(F2 or F4) = 0.37 + 0.315 = 0.685, with no overlap.

**Interpretation.** Almost every building (73.5%) is either large or needs extra AHUs, and only 53 are neither. The overlap term is large (0.485) because the two events are strongly linked: nearly all large buildings need extra AHUs (the dependence is studied in Applications 6-10). Adding 0.49 + 0.73 would give 1.22, which cannot be a probability, showing why the overlap must be subtracted. For floors the buildings fall into exactly one group, so nothing is subtracted.

## 5.4 The rejected seeds as a second probability model (Member 2)

### Application 5 - Why do seeds fail? Conditional probabilities inside R

**Question.** Given that a seed was rejected, how bad was the failure?

**Events and assumptions.** The sample space is now the 82 rejected seeds, which is the event R from Application 1. Every probability below is therefore conditional on R: P(E | R) = |E and R| / |R|. E1 = exactly 1 undersized zone; E2 = smallest zone under 5 m2; E3 = 3 or more undersized zones. (An "undersized" zone is below the generator's 20 m2 minimum.) These counts were parsed from the rejection text by Member 1.

**Rule and why it fits.** Classical probability on the restricted sample space. This is the first use of conditioning in the notebook, and it fits because the rejected-seed file contains only failures; mixing in the accepted buildings would hide the failure pattern.

In [ ]:
R_n = len(rej)
E1 = rej.n_small_zones == 1
E2 = rej.smallest_m2 < 5
E3 = rej.n_small_zones >= 3

tbl = pd.DataFrame({'event': ['E1: exactly 1 undersized zone', 'E2: smallest zone < 5 m2', 'E3: 3+ undersized zones'],
                    'count': [int(E1.sum()), int(E2.sum()), int(E3.sum())]})
tbl['P(E | R)'] = (tbl['count'] / R_n).round(3)
display(tbl)

dist = rej.n_small_zones.value_counts().sort_index()
print('Distribution of n_small_zones:', dist.to_dict())
print(f'mean = {rej.n_small_zones.mean():.2f}, max = {rej.n_small_zones.max()}')
assert np.isclose(dist.sum(), R_n) and np.isclose((dist / R_n).sum(), 1)   # valid distribution again

# E1 and E3 are disjoint, E1 and E2 overlap -> addition rule on the second model
p_E1orE3 = E1.mean() + E3.mean()
assert np.isclose(p_E1orE3, (E1 | E3).mean())
print(f'P(E1 or E3 | R) = {p_E1orE3:.3f}  (disjoint, so we add)')
print(f'P(E1 or E2 | R) = {E1.mean() + E2.mean() - (E1 & E2).mean():.3f}  (overlap {(E1 & E2).sum()} subtracted)')

# How close were the failures to the 20 m2 limit?
single = rej[E1]
print(f'Single-zone failures: median smallest zone = {single.smallest_m2.median():.1f} m2; '
      f'{int((single.smallest_m2 >= 15).sum())} of {len(single)} were within 5 m2 of the limit')

m2_results.update({'P(E1|R)': E1.mean(), 'P(E2|R)': E2.mean(), 'P(E3|R)': E3.mean(),
                   'mean n_small_zones': rej.n_small_zones.mean()})

fig, ax = plt.subplots(figsize=(5.6, 3.6))
bars = ax.bar(dist.index.astype(str), dist.values, color=PALETTE['accent'], width=0.6)
for b, v in zip(bars, dist.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.8, str(v), ha='center', fontsize=10)
ax.set_xlabel('Undersized zones in the rejected seed'); ax.set_ylabel('Number of rejected seeds')
ax.set_title('Rejected seeds: how many zones failed the 20 m2 rule?')
plt.tight_layout(); plt.savefig('figures/fig_rejected_small_zones.png'); plt.show()

**Result.** P(E1 | R) = 54/82 = 0.659, P(E2 | R) = 26/82 = 0.317, P(E3 | R) = 11/82 = 0.134. The mean is 1.59 undersized zones per rejected seed (maximum 7).

**Interpretation.** Two out of three failures are caused by a single undersized zone, so for most rejected seeds a single zone is what separates them from acceptance, and only a small share (13%) are badly broken (3+ zones). Single-zone failures are not all near-misses, though: the median offending zone is 10 m2, half the 20 m2 limit, and only 12 of the 54 were within 5 m2 of it. Caution on the remedy: the generator's own rejection message says the minimum room size parameter does not control the realised minimum and recommends **changing the seed**. So the practical action for organizers is to redraw a seed, not to adjust a parameter; this is exactly what the run did, and it costs about 1.4 tries per usable building (1/0.709).

**Limits.** These are counts over 82 rejected seeds of one generator run. They describe how the generator fails, not real buildings.

In [ ]:
# Hand-off: rule-based results for Member 4's validation table (and a copy on disk)
m2_results = {k: (float(v) if isinstance(v, (float, np.floating)) else int(v)) for k, v in m2_results.items()}
with open('m2_results.json', 'w') as f:
    json.dump(m2_results, f, indent=2)
pd.Series(m2_results, name='value').to_frame()

### 5.5 Conditional probability (Member 3)

**Question.** Are large buildings more likely to need extra air handlers (AHUs) than smaller buildings?

**Events and assumptions.**
- `A` = the building is large: `n_zones > 64` (shared column `A_large`).
- `B` = the building needed at least one AHU beyond its configured number: `ahus_added_beyond_config > 0` (shared column `B_extra_ahu`).
- The 200 accepted buildings are treated as a finite population of equally likely outcomes, so every probability is a count ratio. *Conditioning on `A` means shrinking the sample space to the 98 rows where `A` is true.*
- All results are conditional on the generator having accepted the seed (see Section 3).

**Rule and why it fits.** The definition of conditional probability,

$$P(B \mid A) = \frac{P(A \cap B)}{P(A)} = \frac{N(A \cap B)}{N(A)}, \qquad P(A) > 0.$$

The question compares the extra-AHU rate *inside* two sub-populations (large vs. not large), which is exactly what a conditional probability measures; the unconditional `P(B)` would hide that difference. Direction matters: `P(B | A)` ("among large buildings, how many need extra AHUs?") is **not** `P(A | B)` ("among buildings with extra AHUs, how many are large?"). The second one is answered in Section 5.8.

In [ ]:
# --- 5.5 Conditional probability -------------------------------------------
import matplotlib.pyplot as plt
from math import isclose

def Pc(a, b):
    '''P(a | b) for two Boolean Series: count(a and b) / count(b).'''
    return (a & b).sum() / b.sum()

A, Bx = df.A_large, df.B_extra_ahu          # shared event columns from Section 4
n = len(df)

# 2 x 2 table of counts: the whole sample space split by A and B
counts_AB = pd.crosstab(
    A.map({True: 'A: large (n_zones > 64)', False: 'not A: not large'}),
    Bx.map({True: 'B: extra AHU', False: 'not B: no extra AHU'}),
    margins=True, margins_name='Total')
display(counts_AB)

# Route 1: definition  P(B|A) = P(A and B) / P(A)
P_A, P_B, P_AB = A.mean(), Bx.mean(), (A & Bx).mean()
P_B_given_A = P_AB / P_A
P_B_given_notA = (~A & Bx).mean() / (~A).mean()

# Route 2: restrict the rows first, then take the share (reduced sample space)
assert isclose(P_B_given_A, Bx[A].mean())
assert isclose(P_B_given_notA, Bx[~A].mean())
assert isclose(P_B_given_A, Pc(Bx, A)) and isclose(P_B_given_notA, Pc(Bx, ~A))

print(f"P(A)         = {A.sum()}/{n} = {P_A:.4f}")
print(f"P(B)         = {Bx.sum()}/{n} = {P_B:.4f}")
print(f"P(A and B)   = {(A & Bx).sum()}/{n} = {P_AB:.4f}")
print(f"P(B | A)     = {(A & Bx).sum()}/{A.sum()} = {P_B_given_A:.4f}   (extra AHU, among LARGE buildings)")
print(f"P(B | not A) = {(~A & Bx).sum()}/{(~A).sum()} = {P_B_given_notA:.4f}   (extra AHU, among NOT-large buildings)")
print(f"Difference   = {P_B_given_A - P_B_given_notA:.4f};  P(B|A) / P(B) = {P_B_given_A / P_B:.3f}")

# Sanity check against the numbers quoted in the interpretation below
assert (round(P_B_given_A, 4), round(P_B_given_notA, 4), round(P_B, 4)) == (0.9898, 0.4804, 0.73)

# Does the picture depend on the 64-zone threshold? (local masks: df is not modified)
rows = []
for t in (40, 50, 64, 80, 100):
    big = df.n_zones > t
    rows.append({'large means n_zones >': t, 'N(large)': int(big.sum()),
                 'P(B | large)': Pc(Bx, big), 'P(B | not large)': Pc(Bx, ~big)})
threshold_check = pd.DataFrame(rows)
display(threshold_check.round(3))

# Why? Look at the generator's own AHU rule (all checked on the data, not assumed)
need_more = df.min_ahus_required > df.ahus_configured
assert (need_more == Bx).all()                                   # B  <=>  required AHUs > configured AHUs
assert (df.min_ahus_required == np.ceil(df.n_zones / 10)).all()  # required AHUs = ceil(n_zones / 10)
big70 = df.n_zones > 70
print("\nrequired AHUs = ceil(n_zones/10) in every row; configured AHUs range", df['ahu.num_ahus'].min(), "to", df['ahu.num_ahus'].max())
print(f"P(B | n_zones > 70) = {Pc(Bx, big70):.3f}  ({int(big70.sum())} buildings)")
print("Large buildings WITHOUT extra AHU:")
display(df.loc[A & ~Bx, ['seed', 'n_zones', 'ahu.num_ahus', 'min_ahus_required', 'floor_plan.num_floors']])

# Chart: unconditional vs conditional extra-AHU probability
fig, ax = plt.subplots(figsize=(6.5, 4))
labels = ['P(B)\nall buildings', 'P(B | A)\nlarge', 'P(B | not A)\nnot large']
vals = [P_B, P_B_given_A, P_B_given_notA]
bars = ax.bar(labels, vals, color=['#7f7f7f', '#d95f02', '#1b9e77'])
for b_, v in zip(bars, vals):
    ax.text(b_.get_x() + b_.get_width() / 2, v + 0.015, f'{v:.3f}', ha='center', fontsize=11)
ax.set_ylim(0, 1.1); ax.set_ylabel('Probability of an extra AHU')
ax.set_title('Knowing the building is large changes P(extra AHU)')
plt.tight_layout(); plt.show()

**Interpretation (5.5).** Among the 98 large buildings, 97 needed extra AHUs: `P(B | A) = 97/98 = 0.990`. Among the 102 not-large buildings only 49 did: `P(B | not A) = 49/102 = 0.480`. The unconditional `P(B) = 0.730` sits in between because it averages the two groups. Learning that a building is large raises the chance of an extra AHU by about 51 percentage points, so size is a very strong predictor of `B`.

The data shows why: the generator requires `ceil(n_zones / 10)` AHUs and only configures 3 to 7 of them, so a building has an extra AHU exactly when the required number exceeds the configured number. Any building with more than 70 zones needs at least 8 AHUs and therefore *always* has extra ones (`P = 1.000`); the single large building without an extra AHU has 65 zones and 7 configured AHUs. The gap between large and not-large remains big for every threshold in the table, so it is not an artefact of choosing the median. This is a mechanical rule of the generator, i.e. an association in generated data, not a causal finding about real buildings.

### 5.6 Multiplication rule and chain rule (Member 3)

**Question.** What share of *all* buildings are both large and need extra AHUs, and can we build that joint probability from a marginal and a conditional probability? Can the same idea handle three events (4 floors, large, extra AHU)?

**Events and assumptions.** `A` and `B` as in 5.5; `F4` = the building has 4 floors (shared column `F4`). Same finite-population, equally-likely-rows assumption.

**Rule and why it fits.** The multiplication rule rearranges the definition of conditional probability:

$$P(A \cap B) = P(A)\,P(B \mid A) = P(B)\,P(A \mid B).$$

It is the right tool because we already know a conditional probability (`P(B | A)`) and want the joint probability. The plain product `P(A) P(B)` would be correct only if `A` and `B` were independent, which the next calculation shows is *not* the case. For three events the chain rule extends it:

$$P(F_4 \cap A \cap B) = P(F_4)\,P(A \mid F_4)\,P(B \mid F_4 \cap A).$$

In [ ]:
# --- 5.6 Multiplication rule and chain rule --------------------------------
F4 = df.F4

# Two events: P(A and B) three ways
joint_rule_1 = P_A * Pc(Bx, A)                 # P(A) * P(B|A)
joint_rule_2 = P_B * Pc(A, Bx)                 # P(B) * P(A|B)   (other order, same answer)
joint_direct = (A & Bx).mean()                 # direct count: 97 / 200
joint_if_independent = P_A * P_B               # what the plain product would give

print(f"P(A) P(B|A)             = {P_A:.4f} x {Pc(Bx, A):.4f} = {joint_rule_1:.4f}")
print(f"P(B) P(A|B)             = {P_B:.4f} x {Pc(A, Bx):.4f} = {joint_rule_2:.4f}")
print(f"direct count (A & B)    = {(A & Bx).sum()}/{n} = {joint_direct:.4f}")
print(f"P(A) P(B) if independent = {joint_if_independent:.4f}   <- does NOT match, so A and B are dependent")
assert isclose(joint_rule_1, joint_direct) and isclose(joint_rule_2, joint_direct)

# Three events: chain rule, each factor read from the data
p1 = F4.mean()                  # P(F4)
p2 = Pc(A, F4)                  # P(A | F4)
p3 = Pc(Bx, F4 & A)             # P(B | F4 and A)
chain = p1 * p2 * p3
chain_direct = (F4 & A & Bx).mean()
print(f"\nP(F4)         = {F4.sum()}/{n} = {p1:.4f}")
print(f"P(A | F4)     = {(F4 & A).sum()}/{F4.sum()} = {p2:.4f}")
print(f"P(B | F4 & A) = {(F4 & A & Bx).sum()}/{(F4 & A).sum()} = {p3:.4f}")
print(f"Chain rule    = {p1:.4f} x {p2:.4f} x {p3:.4f} = {chain:.4f}")
print(f"Direct count  = {(F4 & A & Bx).sum()}/{n} = {chain_direct:.4f}")
assert isclose(chain, chain_direct)

multiplication_table = pd.DataFrame({
    'quantity': ['P(A and B)', 'P(F4 and A and B)'],
    'rule-based': [joint_rule_1, chain],
    'direct count': [joint_direct, chain_direct],
    'count / 200': [f"{(A & Bx).sum()}/{n}", f"{(F4 & A & Bx).sum()}/{n}"]})
display(multiplication_table.round(4))

**Interpretation (5.6).** About 48.5% of all accepted buildings are both large and need extra AHUs (`0.49 x 0.990 = 0.485 = 97/200`). If size and the extra-AHU need were independent we would expect only `0.49 x 0.73 = 0.358`, so the multiplication rule with the *conditional* factor is the correct way to get the joint probability here; the gap between 0.358 and 0.485 is itself evidence of dependence (formally assessed in Section 5.9 by Member 4). The chain rule shows the structure of the biggest group: 31.5% of buildings have 4 floors, 79.4% of those are large, and *every* 4-floor large building needs extra AHUs (`P(B | F4 and A) = 1`), giving 25% of all buildings. The rule-based and direct values agree exactly because both use the same 200-row population.

### 5.7 Total probability (Member 3)

**Question.** Can we recover the overall chance of an extra AHU, `P(B)`, from the floor-level extra-AHU rates and the floor mix alone, without ever counting `B` over all 200 rows?

**Events and assumptions.** The partition is `F2, F3, F4` (2, 3 or 4 floors). For the law to apply the events must be **mutually exclusive** and **exhaustive**, and `P(F_k) > 0`. This is checked in code (every building has exactly one floor count; 74 + 63 + 63 = 200). A second partition, `A` / `not A`, is used as a cross-check.

**Rule and why it fits.** The law of total probability:

$$P(B) = \sum_{k=2}^{4} P(B \mid F_k)\,P(F_k).$$

It fits because the extra-AHU rate clearly differs by floor count (we expect it to rise with floors), so `P(B)` is a weighted average of the group rates with the share of each group as weight. It is also the denominator needed for Bayes' theorem in Section 5.8.

In [ ]:
# --- 5.7 Total probability ---------------------------------------------------
# 1) Check that F2, F3, F4 form a partition
floor_events = {k: df[f'F{k}'] for k in (2, 3, 4)}
assert ((df.F2.astype(int) + df.F3.astype(int) + df.F4.astype(int)) == 1).all()   # exactly one per row
assert sum(int(f.sum()) for f in floor_events.values()) == n                       # 74 + 63 + 63 = 200

# 2) Ingredients: P(F_k) and P(B | F_k)
pF   = {k: df[f'F{k}'].mean() for k in (2, 3, 4)}
pB_F = {k: Pc(Bx, df[f'F{k}']) for k in (2, 3, 4)}

# 3) Total probability
pB_total = sum(pB_F[k] * pF[k] for k in (2, 3, 4))

total_prob_table = pd.DataFrame({
    'floors': [2, 3, 4],
    'N(F_k)': [int(floor_events[k].sum()) for k in (2, 3, 4)],
    'P(F_k)': [pF[k] for k in (2, 3, 4)],
    'N(B and F_k)': [int((floor_events[k] & Bx).sum()) for k in (2, 3, 4)],
    'P(B | F_k)': [pB_F[k] for k in (2, 3, 4)],
    'P(B | F_k) x P(F_k)': [pB_F[k] * pF[k] for k in (2, 3, 4)]})
display(total_prob_table.round(4))
print(f"Total probability  P(B) = {' + '.join(f'{pB_F[k] * pF[k]:.3f}' for k in (2, 3, 4))} = {pB_total:.4f}")
print(f"Direct count       P(B) = {Bx.sum()}/{n} = {Bx.mean():.4f}")
assert isclose(pB_total, Bx.mean())

# 4) Cross-check with a different partition: A and not A
pB_total_A = P_A * Pc(Bx, A) + (1 - P_A) * Pc(Bx, ~A)
print(f"Partition A / not A: {P_A:.3f} x {Pc(Bx, A):.4f} + {1 - P_A:.3f} x {Pc(Bx, ~A):.4f} = {pB_total_A:.4f}")
assert isclose(pB_total_A, Bx.mean())

# 5) Joint probability table floors x extra AHU (the pieces of the law; reusable for a heatmap)
joint_F_B = pd.crosstab(df['floor_plan.num_floors'], Bx, normalize='all')
joint_F_B.columns = ['no extra AHU (not B)', 'extra AHU (B)']
joint_F_B.index.name = 'floors'
display(joint_F_B.round(4))
assert isclose(joint_F_B['extra AHU (B)'].sum(), pB_total)    # column sum = P(B)

**Interpretation (5.7).** The extra-AHU rate climbs with the number of floors: `P(B | F2) = 39/74 = 0.527`, `P(B | F3) = 49/63 = 0.778`, `P(B | F4) = 58/63 = 0.921`. Weighting these rates by the floor mix (0.370, 0.315, 0.315) gives `0.195 + 0.245 + 0.290 = 0.730`, exactly the direct count `146/200`. The agreement is exact (not approximate) because the same 200 buildings feed both routes and the partition is valid; a mismatch would signal an overlapping or incomplete partition. The `A` / `not A` partition gives the same 0.730, showing that the result does not depend on which partition is used. More floors means more zones in this generator, which is why taller buildings more often need extra AHUs; the overall 73% is a blend of a roughly 50/50 situation for 2-floor buildings and a near-certain one for 4-floor buildings.

### 5.8 Bayes' theorem (Member 3)

**Question.** Suppose we are told only that a building needed extra AHUs. How likely is it to have 4 floors (and how likely is it to be large)? Conversely, how does the answer change if we are told it needed *no* extra AHU?

**Events and assumptions.** Prior: `P(F_k)` (floor mix before any evidence). Evidence: `B` (extra AHU observed). Posterior: `P(F_k | B)`. We know the forward rates `P(B | F_k)` from 5.7 and want to reverse the direction of the conditioning. Finite population of 200 accepted buildings, as before.

**Rule and why it fits.** Bayes' theorem:

$$P(F_k \mid B) = \frac{P(B \mid F_k)\,P(F_k)}{P(B)}, \qquad P(B) = \sum_j P(B \mid F_j)\,P(F_j).$$

It fits because the data naturally give "given the floors, how often is there an extra AHU?" but the question asks the reverse, "given an extra AHU, what are the floors?". The denominator is the total-probability result from 5.7, so 5.7 and 5.8 work as a pair. The same formula answers `P(A | B) = P(B | A) P(A) / P(B)`.

In [ ]:
# --- 5.8 Bayes' theorem ------------------------------------------------------
# Posterior P(F_k | B) for every floor count (evidence: an extra AHU was needed)
post = {k: pB_F[k] * pF[k] / pB_total for k in (2, 3, 4)}
post_F4 = post[4]
direct_post = {k: Pc(df[f'F{k}'], Bx) for k in (2, 3, 4)}        # direct count among B rows
for k in (2, 3, 4):
    assert isclose(post[k], direct_post[k])
assert isclose(sum(post.values()), 1.0)                          # posteriors form a distribution

# Evidence in the other direction: no extra AHU (not B)
pnotB = 1 - pB_total
post_notB = {k: (1 - pB_F[k]) * pF[k] / pnotB for k in (2, 3, 4)}
for k in (2, 3, 4):
    assert isclose(post_notB[k], Pc(df[f'F{k}'], ~Bx))

prior_posterior = pd.DataFrame({
    'floors': [2, 3, 4],
    'prior P(F_k)': [pF[k] for k in (2, 3, 4)],
    'likelihood P(B | F_k)': [pB_F[k] for k in (2, 3, 4)],
    'posterior P(F_k | B)': [post[k] for k in (2, 3, 4)],
    'direct count among B': [f"{int((df[f'F{k}'] & Bx).sum())}/{int(Bx.sum())}" for k in (2, 3, 4)],
    'posterior P(F_k | not B)': [post_notB[k] for k in (2, 3, 4)]})
display(prior_posterior.round(4))
print(f"P(F4 | B) = ({pB_F[4]:.4f} x {pF[4]:.3f}) / {pB_total:.3f} = {post_F4:.4f}   direct: 58/146 = {58 / 146:.4f}")

# Second Bayes question: P(A | B), "of the buildings with extra AHUs, how many are large?"
post_A_given_B = Pc(Bx, A) * P_A / P_B
print(f"P(A | B) = ({Pc(Bx, A):.4f} x {P_A:.2f}) / {P_B:.2f} = {post_A_given_B:.4f}   direct: 97/146 = {97 / 146:.4f}   prior P(A) = {P_A:.2f}")
assert isclose(post_A_given_B, Pc(A, Bx)) and isclose(post_F4, 58 / 146)

# Direction check: the two conditional probabilities are very different
print(f"\nP(B | A) = {Pc(Bx, A):.3f}  vs  P(A | B) = {Pc(A, Bx):.3f}   (not the same question!)")

# Chart 1: prior vs posterior
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(3); w = 0.27
ax.bar(x - w, [pF[k] for k in (2, 3, 4)], w, label='prior P(F_k)', color='#7f7f7f')
ax.bar(x, [post[k] for k in (2, 3, 4)], w, label='posterior P(F_k | extra AHU)', color='#d95f02')
ax.bar(x + w, [post_notB[k] for k in (2, 3, 4)], w, label='posterior P(F_k | no extra AHU)', color='#1b9e77')
for xs, vals in ((x - w, [pF[k] for k in (2, 3, 4)]), (x, [post[k] for k in (2, 3, 4)]), (x + w, [post_notB[k] for k in (2, 3, 4)])):
    for xi, v in zip(xs, vals):
        ax.text(xi, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(['2 floors', '3 floors', '4 floors'])
ax.set_ylabel('Probability'); ax.set_ylim(0, 0.8)
ax.set_title('How the evidence "extra AHU" updates the floor distribution')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

# Chart 2: probability tree floors -> extra AHU (branch probabilities and joint probabilities)
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
arrow = dict(arrowstyle='->', color='#444444')
ax.text(0.06, 0.5, 'All 200\naccepted\nbuildings', ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round,pad=0.4', fc='#e0e0e0'))
floor_y = {2: 0.82, 3: 0.50, 4: 0.18}
for k in (2, 3, 4):
    fy = floor_y[k]
    ax.annotate('', xy=(0.31, fy), xytext=(0.12, 0.5), arrowprops=arrow)
    ax.text(0.20, (0.5 + fy) / 2 + (0.05 if k == 2 else -0.05 if k == 4 else 0.03),
            f'P(F{k}) = {pF[k]:.3f}', ha='center', fontsize=10)
    ax.text(0.37, fy, f'{k} floors', ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.4', fc='#cfe2f3'))
    for is_b, dy, lab, col in ((True, 0.09, 'extra AHU (B)', '#d95f02'), (False, -0.09, 'no extra AHU (not B)', '#1b9e77')):
        ly = fy + dy
        p_cond = pB_F[k] if is_b else 1 - pB_F[k]
        ax.annotate('', xy=(0.60, ly), xytext=(0.43, fy), arrowprops=arrow)
        ax.text(0.515, (fy + ly) / 2 + (0.035 if is_b else -0.035), f'{p_cond:.3f}', ha='center', fontsize=10)
        ax.text(0.62, ly, f'{lab}:  joint = {pF[k] * p_cond:.3f}', va='center', fontsize=10, color=col)
ax.set_title('Probability tree: floors -> extra AHU (branch = conditional, end = joint probability)', fontsize=12)
plt.tight_layout(); plt.show()

**Interpretation (5.8).** *Prior* = what we believed about the floor count before seeing any evidence (31.5% four-floor); *posterior* = the updated belief after learning the building needed extra AHUs. Seeing that evidence raises the probability of 4 floors from 0.315 to `(58/63 x 0.315) / 0.73 = 0.397` (`= 58/146`, matching the direct count) and lowers the probability of 2 floors from 0.370 to 0.267. Hearing "no extra AHU" pushes the other way: the 4-floor share falls to 0.093 (5 of 54 buildings) and 2-floor buildings become the majority (0.648). In the same way, among buildings with extra AHUs about two thirds are large (`P(A | B) = 97/146 = 0.664`, up from the prior 0.49).

Two cautions. First, `P(B | A) = 0.990` and `P(A | B) = 0.664` answer different questions; the first is high because almost all large buildings need extra AHUs, the second is lower because many *not-large* buildings (49 of 146) also need them. Second, the posterior is a re-weighting of the observed buildings, not a causal statement: the extra AHU is "evidence" about the floors only because the generator links floors, size and AHU requirements.

#### Hand-off to Member 4 (validation): rule-based results from Sections 5.5-5.8

All values below are pulled live from the cells above (nothing is typed in by hand). Member 4's validation cells can use the variables `Pc`, `A`, `Bx`, `pF`, `pB_F`, `pB_total`, `post_F4` and the dictionary `m3_results`.

In [ ]:
# --- Hand-off dictionary: rule-based values (route 1 of the three-way validation) ---
m3_results = {
    'P(B|A)':          P_B_given_A,
    'P(B|not A)':      P_B_given_notA,
    'P(A and B)':      joint_rule_1,                 # multiplication rule
    'P(F4 and A and B)': chain,                      # chain rule
    'P(B)':            pB_total,                     # total probability
    'P(F4|B)':         post_F4,                      # Bayes
    'P(A|B)':          post_A_given_B,               # Bayes
    'P(F2|B)':         post[2],
    'P(F3|B)':         post[3],
}
display(pd.Series(m3_results, name='rule-based value').round(4).to_frame())

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from math import comb, perm, isclose
from pathlib import Path
from scipy.stats import chi2_contingency
from IPython.display import display

EVENTS = ["A_large", "B_extra_ahu", "F2", "F3", "F4", "O2"]

def rebuild_member1_df():
    """Same steps as Member 1 (Sections 1, 3, 4). Used only if df is not already in memory."""
    need = ["buildings_200_features.csv", "rejected_seeds.csv", "feature_catalog.csv"]
    folders = [Path.cwd(), Path("/content"), Path("/content/drive/MyDrive/stats_project")]
    data_dir = next((p for p in folders if (p / need[0]).exists()), None)
    if data_dir is None:
        from google.colab import files
        print("Upload buildings_200_features.csv now (rejected_seeds / feature_catalog are optional here)")
        files.upload(); data_dir = Path.cwd()
    raw = pd.read_csv(data_dir / need[0])                      # raw table is never modified
    d = raw.copy()
    d.columns = [c.split("__", 1)[1] for c in d.columns]        # drop only the first prefix
    d = d.drop(columns=[c for c in d.columns if d[c].nunique(dropna=False) == 1])   # 17 constants
    d["A_large"]     = d.n_zones > d.n_zones.median()           # strict >, median = 64
    d["B_extra_ahu"] = d.ahus_added_beyond_config > 0
    for k in (2, 3, 4):
        d[f"F{k}"] = d["floor_plan.num_floors"] == k
    d["O2"] = d["simulation.occupants_per_zone"] == 2
    return raw, d

if "df" not in globals() or not all(e in df.columns for e in EVENTS):
    df_raw, df = rebuild_member1_df()
    print("Stand-alone mode: rebuilt df from the CSV")
else:
    print("Using Member 1's df from this notebook")

# --- Member 4's own additions (df_raw untouched; N7 added to the analysis frame only) ---
med = df.n_zones.median()                          # 64
df["N7"] = df["ahu.num_ahus"] == 7                 # 7 AHUs were configured
cut = df.n_zones.quantile(0.95)                    # 157.7 zones
df_trim = df[df.n_zones <= cut].copy()             # 5% of buildings with the most zones removed
n = len(df)

print("df:", df.shape, "| df_trim:", df_trim.shape, "| cut =", round(cut, 1))
counts = df[EVENTS + ["N7"]].sum()
print(counts.to_dict())
assert n == 200 and len(df_trim) == 190
assert counts.to_dict() == {"A_large": 98, "B_extra_ahu": 146, "F2": 74, "F3": 63,
                            "F4": 63, "O2": 108, "N7": 39}, "event counts differ from Member 1 - check df"

## 5.9 Independence

**Question.** Does knowing one fact about a building change the chance of another fact?

**Events and assumptions.** `A_large` (n_zones > 64, 98 buildings), `B_extra_ahu` (AHUs added beyond configured, 146), `F2/F3/F4` (2/3/4 floors: 74/63/63), `O2` (2 occupants per zone, 108), `N7` (7 AHUs configured, 39). The 200 accepted buildings are treated as equally likely draws from the generator; tolerance is 2 standard errors.

**Rule and why it fits.** Two events are independent if P(A and B) = P(A) x P(B). We compare the actual joint probability with this product. With only 200 rows the gap is never exactly 0, so we call a pair *dependent* only when the gap is bigger than 2 SE of the gap, where SE = sqrt( P(A)(1-P(A)) P(B)(1-P(B)) / n ). A chi-square p-value is a cross-check. Lift = P(A and B) / (P(A)P(B)); 1 means independent, above 1 means they occur together more than chance, below 1 less than chance.

In [ ]:
def indep(a, b, name):
    pa, pb, pab = a.mean(), b.mean(), (a & b).mean()
    pp = pa * pb                                                   # expected joint if independent
    se = (pa*(1-pa)*pb*(1-pb) / n) ** 0.5                          # standard error of the gap
    p_chi2 = chi2_contingency(pd.crosstab(a, b), correction=False)[1]   # cross-check only
    return dict(pair=name, P_A=pa, P_B=pb, P_AandB=pab, P_A_x_P_B=pp, diff=pab-pp,
                lift=pab/pp, two_SE=2*se,
                verdict="dependent" if abs(pab-pp) > 2*se else "consistent with independence",
                p_chi2=p_chi2)

pairs = [("A_large","B_extra_ahu"), ("F4","B_extra_ahu"), ("O2","B_extra_ahu"),
         ("F4","O2"), ("F4","N7"), ("O2","A_large"),
         ("F2","B_extra_ahu"), ("F3","B_extra_ahu"), ("F4","A_large"),
         ("N7","B_extra_ahu"), ("N7","A_large"), ("O2","N7")]
indep_table = pd.DataFrame([indep(df[a], df[b], f"{a} vs {b}") for a, b in pairs])
display(indep_table.round(4))

dep = set(indep_table.loc[indep_table.verdict == "dependent", "pair"])
assert dep == {"A_large vs B_extra_ahu", "F4 vs B_extra_ahu", "F2 vs B_extra_ahu",
               "F4 vs A_large", "N7 vs B_extra_ahu"}, dep
print("Dependent pairs:", sorted(dep))

**Interpretation (5.9).** Five pairs are dependent: A_large/F4/F2/N7 each with the AHU outcome or size. They all trace back to the generator's capacity rule (required AHUs = ceil(zones / 10) versus 3 to 7 configured AHUs). The seven consistent pairs are settings the generator draws separately (floors, occupants, configured AHUs) or things outside the AHU rule (occupants). "Consistent with independence" does **not** prove independence; 200 rows cannot prove it. These are associations in generated data, not causes. **Limitation:** the 2 SE rule is a rule of thumb, so we also report the chi-square p-value.

### Lift chart
Lift = P(A and B) / (P(A) x P(B)); the black line at 1 means independent.

In [ ]:
t = indep_table.sort_values("lift")
colors = ["#C8662B" if v == "dependent" else "#2E6F95" for v in t.verdict]
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(t.pair, t.lift, color=colors, height=0.6)
ax.axvline(1, color="black", lw=1)
for y, l in enumerate(t.lift):
    ax.text(l + 0.02, y, f"{l:.2f}", va="center", fontsize=8)
ax.set(title="Lift per event pair (orange = dependent, blue = consistent with independence)",
       xlabel="lift = P(A and B) / (P(A) x P(B))   (1 = independent)")
plt.tight_layout(); plt.savefig("lift_chart.png", dpi=150); plt.show()

## 6. Validation

### 6.1 Three-way validation table

**Question.** Do independent routes give the same probabilities?

**Events and assumptions.** Same 200 rows, seed 42. Monte Carlo draws 100,000 buildings from the 200 *with replacement*.

**Rule and why it fits.** Each result is computed (1) by its probability rule, (2) by directly counting rows, (3) by Monte Carlo. Rule and direct count are the same number written two ways, so they must match to rounding (< 1e-12). Monte Carlo is random, so it differs by sampling noise only (about sqrt(p(1-p)/100000) = 0.0016 at most). If Member 3's `m3_results` is in memory, its stored values are checked against ours as well.

In [ ]:
def Pc(a, b):                       # P(a | b) = count(a and b) / count(b)
    return (a & b).sum() / b.sum()

A, Bx = df.A_large, df.B_extra_ahu
pF  = {k: df[f"F{k}"].mean() for k in (2, 3, 4)}
pBF = {k: Pc(Bx, df[f"F{k}"]) for k in (2, 3, 4)}
pB_total = sum(pBF[k] * pF[k] for k in (2, 3, 4))               # total probability
post_F4  = pBF[4] * pF[4] / pB_total                              # Bayes

rng = np.random.default_rng(42)                                   # seed 42 -> same numbers every run
sim = df.loc[rng.choice(df.index, 100_000)]                       # 100,000 draws with replacement
cx  = pd.crosstab(A, Bx)                                          # another route to P(B|A)
sA, sB = sim.A_large, sim.B_extra_ahu

items = [
 ("P(B) total probability",     pB_total, Bx.mean(), sB.mean()),
 ("P(B|A) conditional",         (A & Bx).mean()/A.mean(), Pc(Bx, A), Pc(sB, sA)),
 ("P(A and B) multiplication",  A.mean()*Pc(Bx, A), (A & Bx).mean(), (sA & sB).mean()),
 ("P(A or B) addition",         A.mean()+Bx.mean()-(A & Bx).mean(), (A | Bx).mean(), (sA | sB).mean()),
 ("P(F4|B) Bayes",              post_F4, Pc(df.F4, Bx), Pc(sim.F4, sB)),
 ("P(not B) complement",        1-Bx.mean(), (~Bx).mean(), (~sB).mean()),
]
val = pd.DataFrame(items, columns=["result", "rule", "direct_count", "monte_carlo"])
val["|rule-direct|"] = (val.rule - val.direct_count).abs()
val["|direct-MC|"]   = (val.direct_count - val.monte_carlo).abs()
display(val.round(5))
assert (val["|rule-direct|"] < 1e-12).all(), "rule and direct count must agree exactly - look for a bug"
assert (val["|direct-MC|"] < 0.005).all(), "Monte Carlo too far from direct count"
print("crosstab route P(B|A):", round(cx.loc[True, True] / cx.loc[True].sum(), 4))
print("largest Monte Carlo noise seen:", round(val["|direct-MC|"].max(), 5))

# Optional: compare with Member 3's hand-off values (only if his cells ran in this notebook)
if "m3_results" in globals():
    chk = {"P(B|A)": Pc(Bx, A), "P(A and B)": (A & Bx).mean(), "P(B)": Bx.mean(), "P(F4|B)": Pc(df.F4, Bx)}
    for k, v in chk.items():
        assert isclose(m3_results[k], v, abs_tol=1e-12), f"Member 3 value {k} differs"
    print("Member 3 hand-off values match the direct counts.")
else:
    print("m3_results not in memory (stand-alone mode): Member 3 check skipped.")

**Interpretation (6.1).** Rule and direct count agree exactly (difference 0 up to rounding) because both use the same 200 rows. Monte Carlo differs by at most a few thousandths, which is ordinary sampling noise. If rule and direct count ever differ, look for a wrong denominator, a partition that does not add to 200, different row sets (`df` vs `df_trim`), or different thresholds (`>64` vs `>=64`). **Limitation:** Monte Carlo only re-draws our own 200 rows, so it checks our code, not the real generator.

### 6.2 Counting check (without replacement)

**Question.** If 3 buildings are picked at random without replacement, what is the chance all 3 need extra AHUs?

**Rule and why it fits.** Combinations: C(146,3) / C(200,3), because the order of the 3 buildings does not matter and no building can be picked twice. The shortcut 0.73^3 assumes replacement, so it should be slightly different.

In [ ]:
exact = comb(146, 3) / comb(200, 3)
rng = np.random.default_rng(42)
b = df.B_extra_ahu.to_numpy()
hits = sum(b[rng.choice(200, 3, replace=False)].all() for _ in range(100_000))
print(f"exact C(146,3)/C(200,3)          = {exact:.4f}")
print(f"simulation (no replacement)      = {hits/100_000:.4f}")
print(f"with-replacement shortcut 0.73^3 = {0.73**3:.4f}  (gap {0.73**3-exact:+.4f})")
print("ordered triples 200P3 =", perm(200, 3), "| unordered C(200,3) =", comb(200, 3))
assert abs(exact - hits/100_000) < 0.005

**Interpretation (6.2).** The exact value (0.387) and the simulation agree within noise. The with-replacement shortcut is a little too high because drawing without replacement slightly lowers the chance of another B building each time. **Limitation:** the simulation has noise of about 0.002.

### 6.3 Bootstrap (uncertainty)

**Question.** How uncertain is P(B|A) because we only have 200 buildings?

**Rule and why it fits.** Bootstrap: redraw 200 buildings with replacement 2,000 times, recompute P(B|A) each time, and take the middle 95% (2.5th to 97.5th percentile). It needs no formula, which suits a probability that sits near 1.

In [ ]:
rng = np.random.default_rng(42)
boot = []
for _ in range(2000):
    s = df.loc[rng.choice(df.index, len(df))]
    if s.A_large.sum() > 0:
        boot.append(Pc(s.B_extra_ahu, s.A_large))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"P(B|A) = {Pc(Bx, A):.4f}   95% bootstrap interval [{lo:.4f}, {hi:.4f}]   sd = {np.std(boot):.4f}")
plt.figure(figsize=(6, 3.4)); plt.hist(boot, bins=25, color="#2E6F95")
plt.axvline(lo, color="#C8662B", ls="--"); plt.axvline(hi, color="#C8662B", ls="--")
plt.title("Bootstrap distribution of P(B|A)"); plt.xlabel("P(B|A)"); plt.ylabel("count")
plt.tight_layout(); plt.savefig("bootstrap_hist.png", dpi=150); plt.show()
assert lo <= Pc(Bx, A) <= hi

**Interpretation (6.3).** P(B|A) = 97/98 = 0.990, with a 95% interval of about 0.967 to 1.000. The interval touches 1 because many resamples contain no large building without extra AHUs. So the true rate is very likely above 0.96. **Limitation:** the bootstrap only reflects the 200 accepted buildings, not the generator's rejection step.

### 6.4 Sensitivity (do our own choices change the answer?)

**Question.** Does removing the 5% biggest buildings, or moving the "large" threshold, change the main results?

**Rule and why it fits.** We recompute the same probabilities on `df_trim` (190 rows, zones <= 157.7) and for thresholds 50, 64, 80 and 100. Stable results mean the conclusion does not depend on one buildings group or on the median convention.

In [ ]:
print(f"cut = {cut:.1f} zones; rows {len(df)} -> {len(df_trim)}  ({len(df)-len(df_trim)} buildings removed)")
A2, B2 = df_trim.n_zones > med, df_trim.B_extra_ahu               # same threshold (64)
sens = pd.DataFrame({
    "full (200)": [Bx.mean(), Pc(Bx, A), Pc(Bx, ~A), Pc(df.F4, Bx), df.O2.mean()],
    "trimmed":    [B2.mean(), Pc(B2, A2), Pc(B2, ~A2), Pc(df_trim.F4, B2), df_trim.O2.mean()],
}, index=["P(B)", "P(B|A)", "P(B|not A)", "P(F4|B)", "P(O2)"])
sens["change"] = sens["trimmed"] - sens["full (200)"]
display(sens.round(4))
rows = []
for thr in (50, 64, 80, 100):
    a = df.n_zones > thr
    rows.append((thr, a.mean(), Pc(Bx, a), Pc(Bx, ~a)))
thr_table = pd.DataFrame(rows, columns=["zone threshold", "P(large)", "P(B|large)", "P(B|not large)"])
display(thr_table.round(3))

**Interpretation (6.4).** After removing the 10 buildings above 157.7 zones, P(B|A) barely moves (0.990 to 0.989) and P(B) falls by about 1 point (0.730 to 0.716) and P(F4|B) by about 4 points (0.397 to 0.360). At every threshold large buildings are far more likely to need extra AHUs than not-large ones. So the main conclusion does not depend on the 5% largest buildings or on choosing 64. **Limitation:** trimming changes the population, so the probabilities are for "buildings up to 157.7 zones", not for all accepted buildings.

### Save results for Member 5

In [ ]:
member4_results = dict(independence=indep_table, validation=val, boot_ci=(lo, hi), sensitivity=sens, thresholds=thr_table)
indep_table.round(4).to_csv("m4_independence.csv", index=False)
val.round(5).to_csv("m4_validation.csv", index=False)
sens.round(4).to_csv("m4_sensitivity.csv")
print("saved: m4_independence.csv, m4_validation.csv, m4_sensitivity.csv, lift_chart.png, bootstrap_hist.png")
print("Member 5 can read the dictionary member4_results (keys:", list(member4_results), ")")

## 7. Insights and limitations (Member 5)

Summarize findings with numbers, discuss limitations, and explain relevance to the Smart Buildings competition.